In [14]:
from langchain_groq import ChatGroq
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [15]:
# ----------------------------------------------------
# LLM
# ----------------------------------------------------

model = ChatGroq(model="qwen/qwen3.6-27b", groq_api_key=groq_api_key)
model

# ----------------------------------------------------
# Embeddings
# ----------------------------------------------------

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

# ----------------------------------------------------
# Original Documents
# ----------------------------------------------------

documents = [

    Document(
        page_content="""
LangChain is a framework for developing applications powered by large language models.
It provides components such as prompt templates, output parsers, chains, retrievers,
agents, memory, and integrations with many vector databases.
"""
    ),

    Document(
        page_content="""
LangGraph extends LangChain and allows developers to build stateful,
multi-step agent workflows using nodes and edges.
It supports loops, branching, checkpointing, and persistence.
"""
    ),

    Document(
        page_content="""
Chroma is an open-source vector database used to store document embeddings.
It supports similarity search, metadata filtering,
and integrates seamlessly with LangChain.
"""
    )

]

# ----------------------------------------------------
# Text Splitter
# ----------------------------------------------------

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=30
)

In [16]:
split_docs = text_splitter.split_documents(documents)

print(f"Total Chunks: {len(split_docs)}")

# ----------------------------------------------------
# Vector Store
# ----------------------------------------------------

vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings
)

# ----------------------------------------------------
# Retriever
# ----------------------------------------------------

retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 3
    }
)

# ----------------------------------------------------
# Prompt
# ----------------------------------------------------

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are a helpful AI assistant.

Answer the user's question ONLY using the retrieved context.

If the answer is not in the context, say you don't know.

Retrieved Context:
{context}
"""
        ),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}")
    ]
)

Total Chunks: 6


In [17]:
# ----------------------------------------------------
# Document Chain
# ----------------------------------------------------

document_chain = create_stuff_documents_chain(
    model,
    prompt
)

# ----------------------------------------------------
# Retrieval Chain
# ----------------------------------------------------

retrieval_chain = create_retrieval_chain(
    retriever,
    document_chain
)

In [18]:
# ----------------------------------------------------
# Memory Store
# ----------------------------------------------------

store = {}

def get_session_history(
    session_id: str,
) -> BaseChatMessageHistory:

    if session_id not in store:
        store[session_id] = ChatMessageHistory()

    return store[session_id]

# ----------------------------------------------------
# Conversational Retrieval Chain
# ----------------------------------------------------

conversation_chain = RunnableWithMessageHistory(
    retrieval_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

d:\Anaconda3\envs\LangChain_Langgraph_Course\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [19]:
# ----------------------------------------------------
# Query 1
# ----------------------------------------------------

response = conversation_chain.invoke(
    {
        "input": "What is LangGraph?"
    },
    config={
        "configurable": {
            "session_id": "user_1"
        }
    }
)

print(response["answer"])

Error in RootListenersTracer.on_chain_end callback: KeyError('output')



<think>
Here's a thinking process:

1.  **Analyze User Input:** The user asks "What is LangGraph?"
2.  **Analyze Retrieved Context:** The context provided is repeated three times: "LangGraph extends LangChain and allows developers to build stateful, multi-step agent workflows using nodes and edges."
3.  **Match Question to Context:** The question directly asks for a definition/description of LangGraph. The context provides exactly that.
4.  **Formulate Answer:** Use only the provided context. State clearly what LangGraph is based on the text.
   - Draft: LangGraph is a tool that extends LangChain and allows developers to build stateful, multi-step agent workflows using nodes and edges.
5.  **Check Constraints:** "Answer the user's question ONLY using the retrieved context." The draft strictly uses the context. No external info added. Matches perfectly.
6.  **Final Output Generation:** Output the drafted response.✅
</think>

Based on the provided context, LangGraph is a framework that 

In [20]:
# ----------------------------------------------------
# Query 2
# ----------------------------------------------------

response = conversation_chain.invoke(
    {
        "input": "Does it support checkpointing?"
    },
    config={
        "configurable": {
            "session_id": "user_1"
        }
    }
)

print(response["answer"])

Error in RootListenersTracer.on_chain_end callback: KeyError('output')



<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - Question: "Does it support checkpointing?"
   - Key entity/concept: "checkpointing"

2.  **Analyze Retrieved Context:**
   - The context is repeated three times: "It supports loops, branching, checkpointing, and persistence."
   - This directly states that "it" supports checkpointing.

3.  **Formulate Answer:**
   - Check if the answer is in the context: Yes.
   - Construct the answer based *only* on the context: Yes, it supports checkpointing.
   - Ensure it strictly follows the instruction: "Answer the user's question ONLY using the retrieved context."

4.  **Final Output Generation:**
   - "Yes, it supports checkpointing." (Simple, direct, matches context exactly)✅
   - Self-Correction/Verification during thought: The context explicitly says "It supports loops, branching, checkpointing, and persistence." So the answer is a straightforward "Yes." I will state it clearly. All constraints met.✅
   - Output matches re

In [22]:
# ----------------------------------------------------
# Query 2
# ----------------------------------------------------

response = conversation_chain.invoke(
    {
        "input": "What else do you know about?"
    },
    config={
        "configurable": {
            "session_id": "user_1"
        }
    }
)

print(response["answer"])

Error in RootListenersTracer.on_chain_end callback: KeyError('output')



<think>
Here's a thinking process:

1.  **Analyze User Input:** The user asks "What else do you know about?" This is a very open-ended question, but in the context of the provided retrieved context, it likely refers to the topic discussed in the context, which is "Chroma".

2.  **Analyze Retrieved Context:**
   - "Chroma is an open-source vector database used to store document embeddings."
   - "It supports similarity search, metadata filtering,"
   - (Repeats twice, so the information is identical)
   - Key facts: Chroma is an open-source vector database, stores document embeddings, supports similarity search, supports metadata filtering.

3.  **Identify Constraints:**
   - "Answer the user's question ONLY using the retrieved context."
   - "If the answer is not in the context, say you don't know."

4.  **Formulate Response:**
   - The context only provides information about Chroma: it's an open-source vector database for storing document embeddings, and it supports similarity search